In [80]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import model_selection
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn import metrics
from sklearn import ensemble

In [81]:
df = pd.read_csv(r'C:\Users\user\Desktop\Skilfactory\data1\_train_sem09.csv')

In [82]:
df

,Activity,D1,D2,D3,D4,D5,D6,D7,D8,D9,...,D1767,D1768,D1769,D1770,D1771,D1772,D1773,D1774,D1775,D1776
0,1,0.000000,0.497009,0.10,0.0,0.132956,0.678031,0.273166,0.585445,0.743663,...,0,0,0,0,0,0,0,0,0,0
1,1,0.366667,0.606291,0.05,0.0,0.111209,0.803455,0.106105,0.411754,0.836582,...,1,1,1,1,0,1,0,0,1,0
2,1,0.033300,0.480124,0.00,0.0,0.209791,0.610350,0.356453,0.517720,0.679051,...,0,0,0,0,0,0,0,0,0,0
3,1,0.000000,0.538825,0.00,0.5,0.196344,0.724230,0.235606,0.288764,0.805110,...,0,0,0,0,0,0,0,0,0,0
4,0,0.100000,0.517794,0.00,0.0,0.494734,0.781422,0.154361,0.303809,0.812646,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3746,1,0.033300,0.506409,0.10,0.0,0.209887,0.633426,0.297659,0.376124,0.727093,...,0,0,0,0,0,0,0,0,0,0
3747,1,0.133333,0.651023,0.15,0.0,0.151154,0.766505,0.170876,0.404546,0.787935,...,0,0,1,0,1,0,1,0,0,0
3748,0,0.200000,0.520564,0.00,0.0,0.179949,0.768785,0.177341,0.471179,0.872241,...,0,0,0,0,0,0,0,0,0,0
3749,1,0.100000,0.765646,0.00,0.0,0.536954,0.634936,0.342713,0.447162,0.672689,...,0,0,0,0,0,0,0,0,0,0


In [83]:
X, y = df.drop('Activity', axis=1), df['Activity']

In [84]:
y.value_counts(normalize=True)
# Классы сбалансированы

Activity
1    0.542255
0    0.457745
Name: proportion, dtype: float64

In [85]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.3) 

**Создадим модель логистической регрессии**

Основные параметры LogisticRegression:

* random_state — число, на основе которого происходит генерация случайных чисел.
* penalty — метод регуляризации. Возможные значения:
    * 'l1' — L1-регуляризация;
    * 'l2' — L2-регуляризация (используется по умолчанию);
    * 'elasticnet' — эластичная сетка (L1+L2);
    * None — отсутствие регуляризации.
* C — коэффициент обратный коэффициенту регуляризации, то есть равен . Чем больше C, тем меньше регуляризация. По умолчанию C=1, тогда α=1.
* solver — численный метод оптимизации функции потерь logloss, может быть:
    * 'sag' — стохастический градиентный спуск (нужна стандартизация/нормализация);
    * 'saga' — [модификация](https://arxiv.org/pdf/1407.0202.pdf) предыдущего, которая поддерживает работу с негладкими функциями (нужна стандартизация/нормализация);
    * 'newton-cg' — [метод Ньютона с модификацией сопряжённых градиентов](https://docs.scipy.org/doc/scipy/tutorial/optimize.html#newton-conjugate-gradient-algorithm-method-newton-cg) (не нужна стандартизация/нормализация);
    * 'lbfgs' — [метод Бройдена — Флетчера — Гольдфарба — Шанно](https://ru.wikipedia.org/wiki/Алгоритм_Бройдена_—_Флетчера_—_Гольдфарба_—_Шанно) (не нужна стандартизация/нормализация; используется по умолчанию, так как из всех методов теоретически обеспечивает наилучшую сходимость);
    * 'liblinear' — [метод покоординатного спуска](http://www.machinelearning.ru/wiki/index.php?title=Метод_покоординатного_спуска) (не нужна стандартизация/нормализация).
* max_iter — максимальное количество итераций, выделенных на сходимость.

Посмотрим на результаты самого простого исполнения логистической регрессии

In [86]:
# Логистическая регрессия
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train, y_train)

# Делаем предсказания
y_predict_train = log_reg.predict(X_train)
y_predict_test = log_reg.predict(X_test)

# Результаты
print(f'Значение F1-score на тренировочной выборке: {np.round(metrics.f1_score(y_train, y_predict_train), 3)}')
print(f'Значение F1-score на тестовой выборке {np.round(metrics.f1_score(y_test, y_predict_test), 3)}')


Значение F1-score на тренировочной выборке: 0.897
Значение F1-score на тестовой выборке 0.79


Вывод: 
* F1-score на тренировочной выборке 0.897
* F1-score на тестовой выборке 0.79

Модель переобучена, метрики достаточно сильно отличаются на разных выборках.
Оставим гиперпараметры данной модели без изменений и посмотрим на результаты при обучении на 5 фолдах. Будем использовать StratifiedKFold, чтобы сохранить баланс целевой переменной.

In [87]:
# Создаём объект кросс-валидатора 
kf = model_selection.StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

# Считаем метрики на кросс-валидации
cv_metrics = model_selection.cross_validate(
    estimator=log_reg,
    X = X_train,
    y = y_train,
    cv=kf,
    scoring='f1',
    return_train_score=True
)
display(cv_metrics)

print(f'Среднее значение F1-score на тренировочных фолдах: {np.round(np.mean(cv_metrics['train_score']), 3)}')
print(f'Среднее значение F1-score на валидационных фолдах: {np.round(np.mean(cv_metrics['test_score']), 3)}')
print(f'Значение F1-score на тестовой выборке {np.round(metrics.f1_score(y_test, y_predict_test), 3)}')

{'fit_time': array([0.60205245, 0.47995758, 0.5753448 , 0.83085084, 0.58799911]),
 'score_time': array([0.01146626, 0.01096058, 0.01195717, 0.01100421, 0.01095986]),
 'test_score': array([0.76678445, 0.75909879, 0.78863233, 0.77647059, 0.77142857]),
 'train_score': array([0.9085285 , 0.90371025, 0.90652557, 0.91166078, 0.90379523])}

Среднее значение F1-score на тренировочных фолдах: 0.907
Среднее значение F1-score на валидационных фолдах: 0.772
Значение F1-score на тестовой выборке 0.79


Ситуация не изменилась. Значение метрики на валидационных фолдах ниже метрики на тренировочных фолдах, при этом заметим, что валидационная и тестовая выборка показывают примерно одинаковые результаты. Такая модель имеют обощающую способность.

**1.1 GridSearch.**

In [88]:
param_grid = [{'penalty' : ['l1'],
               'solver' : ['liblinear', 'saga'],
               'C' : [0.001, 0.01, 0.1, 1]},
              {'penalty': ['l2'],
               'solver' : ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'],
               'C' : [0.001, 0.01, 0.1, 1]}]

grid_search_1 = GridSearchCV(
    estimator=log_reg,
    param_grid=param_grid,
    cv=kf,
    n_jobs=-1,
    scoring='f1',
    return_train_score=True
    
)
%time grid_search_1.fit(X_train, y_train)


CPU times: total: 625 ms
Wall time: 2min 7s


,estimator,LogisticRegre...ndom_state=42)
,param_grid,"[{'C': [0.001, 0.01, ...], 'penalty': ['l1'], 'solver': ['liblinear', 'saga']}, {'C': [0.001, 0.01, ...], 'penalty': ['l2'], 'solver': ['newton-cg', 'lbfgs', ...]}]"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,penalty,'l2'


In [89]:
# Получаем полные данные о каждой модели
results_G_S_1 = pd.DataFrame(grid_search_1.cv_results_)
best_index = grid_search_1.best_index_

In [90]:
# Считаем метрики
f1_score_train_G_S_1 = np.round(results_G_S_1.loc[best_index, 'mean_train_score'], 3)
f1_score_test_G_S_1 = np.round(results_G_S_1.loc[best_index, 'mean_test_score'], 3)
print(f'Среднее F1-score на тренировочных фолдах лучшей модели: {f1_score_train_G_S_1}')
print(f'Среднее F1-score на валидационных фолдах лучшей модели: {f1_score_test_G_S_1}')
print(f'Наилучшие значения гиперпараметров: {grid_search_1.best_params_}')
print(f'Значение F1-score на тестовой выборке: {np.round(grid_search_1.score(X_test, y_test), 3)}')

Среднее F1-score на тренировочных фолдах лучшей модели: 0.86
Среднее F1-score на валидационных фолдах лучшей модели: 0.778
Наилучшие значения гиперпараметров: {'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}
Значение F1-score на тестовой выборке: 0.794


Значение f1-score на тестовой выборке не изменилось, однако мы можем наблюдать сокращение разрыва между метрикой на валидационной и тренировочной выборке. 
Отметим, что модель затратила 2 min 7 sec.

**1.2 Использование RandomizedSearchCV для поиска оптимальных гиперпараметров**

In [91]:
from sklearn.model_selection import RandomizedSearchCV

params = [{'penalty' : ['l1'],
               'solver' : ['liblinear', 'saga'],
               'C' : [0.001, 0.01, 0.1, 1]},
              {'penalty': ['l2'],
               'solver' : ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'],
               'C' : [0.001, 0.01, 0.1, 1]}]

random_search_1 = RandomizedSearchCV(
    estimator=log_reg,
    param_distributions=params,
    cv=kf,
    n_iter=15,
    n_jobs=-1,
    return_train_score=True,
    random_state=42
)
%time random_search_1.fit(X_train, y_train)

CPU times: total: 8.72 s
Wall time: 45.7 s


,estimator,LogisticRegre...ndom_state=42)
,param_distributions,"[{'C': [0.001, 0.01, ...], 'penalty': ['l1'], 'solver': ['liblinear', 'saga']}, {'C': [0.001, 0.01, ...], 'penalty': ['l2'], 'solver': ['newton-cg', 'lbfgs', ...]}]"
,n_iter,15
,scoring,None
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [92]:
# Получаем полные данные о каждой модели
results_R_S_1 = pd.DataFrame(random_search_1.cv_results_)
best_index = random_search_1.best_index_

f1_score_train_R_S_1 = np.round(results_R_S_1.loc[best_index, 'mean_train_score'], 3)
f1_score_test_R_S_1 = np.round(results_R_S_1.loc[best_index, 'mean_test_score'], 3)
print(f'Среднее F1-score на тренировочных фолдах лучшей модели: {f1_score_train_G_S_1}')
print(f'Среднее F1-score на тестовых фолдах лучшей модели: {f1_score_test_G_S_1}')
print(f'Лучшие гиперпараметры модели: {random_search_1.best_params_}')
print(f'F1-score на тестовом наборе: {np.round(random_search_1.score(X_test, y_test), 3)}')

Среднее F1-score на тренировочных фолдах лучшей модели: 0.86
Среднее F1-score на тестовых фолдах лучшей модели: 0.778
Лучшие гиперпараметры модели: {'solver': 'saga', 'penalty': 'l2', 'C': 0.1}
F1-score на тестовом наборе: 0.767


Использование RandomizedSearchCV для поиска оптимальных гиперпараметров даёт примерно аналогичные результаты, однако мы потратили меньше времени (45 sec).

# Использование продвинутой оптимизации для поиска гиперпараметров.

**1.3 hyperopt**

In [93]:
import hyperopt
from hyperopt import hp, fmin, tpe, Trials

In [94]:
# зададим пространство поиска гиперпараметров
space = {'C': hp.choice('C', [0.0001, 0.001, 0.01, 0.1, 1]),
         'penalty': hp.choice('penalty', ['l1', 'l2']),
         'solver': hp.choice('solver', ['liblinear', 'saga'])}

In [95]:
random_state=42
# Напишем специальную функцию 
def hyperopt_rf(params, cv=kf, X=X_train, y=y_train, random_state=42):
    params = {'C': float(params['C']),
              'penalty' : str(params['penalty']),
              'solver' : str(params['solver'])}
    # Создаём модель
    model = LogisticRegression(**params, random_state=42, max_iter=5000)
    cv_metrics = model_selection.cross_validate(model, X, y, scoring='f1', cv=kf, return_train_score=True, n_jobs=-1)
    
    test_score_mean = np.mean(cv_metrics['test_score'])
    
    score = test_score_mean
    return -score

In [96]:
%%time
trials = Trials() # Для логирования

best=fmin(hyperopt_rf,
          space=space,
          max_evals=20,
          trials=trials,
          rstate=np.random.default_rng(random_state))
print("Наилучшие значения гиперпараметров {}".format(best))

TPE is being used as the default algorithm.


100%|██████████| 20/20 [02:56<00:00,  8.81s/trial, best loss: -0.7782581740567285]
Наилучшие значения гиперпараметров {'C': 3, 'penalty': 1, 'solver': 0}
CPU times: total: 984 ms
Wall time: 2min 56s


In [97]:
log_reg_best = LogisticRegression(C=3, penalty='l2', solver='liblinear', max_iter=5000)
cv_metrics_best = model_selection.cross_validate(estimator=log_reg_best,
                                                 X=X_train,
                                                 y=y_train,
                                                 cv=kf,
                                                 scoring='f1',
                                                 return_train_score=True,
                                                 n_jobs=-1)
display(cv_metrics_best)


{'fit_time': array([0.82834911, 0.86826158, 0.82576251, 0.79038668, 0.90846658]),
 'score_time': array([0.01597023, 0.01194739, 0.01955366, 0.01692629, 0.01093483]),
 'test_score': array([0.7654321 , 0.75874126, 0.78507993, 0.77834179, 0.75222816]),
 'train_score': array([0.93327402, 0.92749779, 0.92743363, 0.93492696, 0.93179805])}

In [ ]:
log_reg_best.fit(X_train, y_train)
y_test_predict = log_reg_best.predict(X_test)
print(f'Среднее значение F1-score на тренировочных фолдах: {np.round(np.mean(cv_metrics_best['train_score']), 3)}')
print(f'Среднее значение F1-score на валидационных фолдах: {np.round(np.mean(cv_metrics_best['test_score']), 3)}')
print(f'Значение F1-score на тестовой выборке: {np.round(metrics.f1_score(y_test, y_test_predict), 3)}')

Значение F1-score на тестовой выборке: 0.788
Среднее значение F1-score на тренировочных фолдах: 0.931
Среднее значение F1-score на валидационных фолдах: 0.768
Значение F1-score на тестовой выборке: 0.788


Как и в предыдущих случаях мы видим переобучение на тренировочных и валидационных фолдах, однако f1-score на независимой тестовой выборке не отклоняется от значений метрики на валидационных фолдах. Так как модель способна обощать данные, то её можно использовать в дальнейшем.

**1.4 Оптимизация с помощью optuna**

In [99]:
import optuna
from sklearn.model_selection import cross_val_score

In [100]:
def optuna_rf(trial):
    # Задаём пространство поиска гиперпараметров
    C = trial.suggest_float('C', 0.0001, 1, log=True)
    solver = trial.suggest_categorical('solver', ['liblinear', 'saga'])
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])
    
    # Модель логистической регрессии
    model = LogisticRegression(random_state=42, max_iter=5000, C=C, solver=solver, penalty=penalty)
    
    scores = cross_val_score(model, X=X_train, y=y_train, cv=kf, scoring='f1', n_jobs=-1)
    score = np.mean(scores)
    
    return score
    
    

In [101]:
study = optuna.create_study(study_name='LogisticRegression', direction='maximize')
study.optimize(optuna_rf, n_trials=50)

[I 2025-10-22 23:26:54,580] A new study created in memory with name: LogisticRegression
[I 2025-10-22 23:26:54,966] Trial 0 finished with value: 0.6966231524436293 and parameters: {'C': 0.00011468399981037537, 'solver': 'saga', 'penalty': 'l1'}. Best is trial 0 with value: 0.6966231524436293.
[I 2025-10-22 23:26:55,321] Trial 1 finished with value: 0.733444892660376 and parameters: {'C': 0.00031032872228600224, 'solver': 'liblinear', 'penalty': 'l2'}. Best is trial 1 with value: 0.733444892660376.
[I 2025-10-22 23:26:55,710] Trial 2 finished with value: 0.745918076726087 and parameters: {'C': 0.000960663807832161, 'solver': 'liblinear', 'penalty': 'l2'}. Best is trial 2 with value: 0.745918076726087.
[I 2025-10-22 23:27:06,250] Trial 3 finished with value: 0.7791125278910676 and parameters: {'C': 0.050233589998788804, 'solver': 'saga', 'penalty': 'l2'}. Best is trial 3 with value: 0.7791125278910676.
[I 2025-10-22 23:27:06,627] Trial 4 finished with value: 0.7746920314269001 and parame

In [102]:
# выводим результаты на обучающей выборке
print(f'Лучшие значения гиперпараметров {study.best_params}')
print(f'f1_score на обучающем наборе: {np.round(study.best_value, 3)}')

Лучшие значения гиперпараметров {'C': 0.04590531545098486, 'solver': 'liblinear', 'penalty': 'l2'}
f1_score на обучающем наборе: 0.78


In [103]:
params_optuna = study.best_params # Передаём параметры
log_reg_best_optuna = LogisticRegression(**params_optuna, random_state=42, max_iter=5000)
# Обучаем модель
log_reg_best_optuna.fit(X_train, y_train)
# Получаем предсказания
y_predict_test_opt = log_reg_best_optuna.predict(X_test)
print(f'Значение F1-score на тестовой выборке: {np.round(metrics.f1_score(y_test, y_predict_test_opt), 3)}')

Значение F1-score на тестовой выборке: 0.794


Результаты использования 4 различных методов оптимизации показывают, что конечный результат на тестовой выборке, которую модель ещё не видела, практически не отличаются, однако каждый метод использует различное количество времени на поиск оптимальных параметров, в нашем случае, самым эффективным и быстрым был метод поиска с помощью библиотеки optuna.

**Сводные результаты по всем моделям:**
* GridSearch. Значение F1-score на тестовой выборке: 0.794 
* RandomSearch. Значение F1-score на тестовой выборке: 0.767
* Hyperopt. Значение F1-score на тестовой выборке: 0.788
* Optuna. Значение F1-score на тестовой выборке: 0.794


Попробуем решить данную проблему с помощью более мощного алгоритма - случайный лес

**Основные параметры RandomForestClassifier:**

* `n_estimators` — количество деревьев в лесу (число K из бэггинга; по умолчанию равно 100);
* `criterion` — критерий информативности разбиения для каждого из деревьев (`'gini'` — критерий Джини и `'entropy'` — энтропия Шеннона; по умолчанию — `'gini'`);
* `max_depth` — максимальная глубина одного дерева (по умолчанию — `None`, то есть глубина дерева не ограничена);
* `max_features` — максимальное число признаков, которые будут использоваться каждым из деревьев (число L из метода случайных подпространств; по умолчанию — `'sqrt'`; для обучения каждого из деревьев используется $\sqrt{m}$ признаков, где $m$ — число признаков в начальном наборе данных);
* `min_samples_leaf` — минимальное число объектов в листе (по умолчанию — 1);
* `random_state` — параметр, отвечающий за генерацию случайных чисел.

Создадим базовую модель, чтобы посмотреть как она решает поставленную задачу.

In [104]:
#Создаём объект класса RandomForestClassifier
random_forest = ensemble.RandomForestClassifier(
    n_estimators=100,
    max_depth=7,
    criterion='gini',
    min_samples_leaf=10,
    random_state=42,
    max_features=400
)

# Используем кросс-валидацию
cv_metrics_random_forest = model_selection.cross_validate(
    estimator=random_forest,
    X=X_train,
    y=y_train,
    scoring='f1',
    return_train_score=True,
    cv=kf
)

display(cv_metrics_random_forest)

print(f'Среднее значение F1-score на тренировочных фолдах: {np.round(np.mean(cv_metrics_random_forest['train_score']), 3)}')
print(f'Среднее значение F1-score на валидационных фолдах: {np.round(np.mean(cv_metrics_random_forest['test_score']), 3)}')

{'fit_time': array([3.33218074, 3.20859742, 3.19746661, 3.23372769, 3.18792772]),
 'score_time': array([0.02110362, 0.01694322, 0.01694345, 0.02003479, 0.01594162]),
 'test_score': array([0.78057554, 0.79649123, 0.7915937 , 0.80550775, 0.79292035]),
 'train_score': array([0.85172109, 0.84995587, 0.84632321, 0.85789474, 0.85165794])}

Среднее значение F1-score на тренировочных фолдах: 0.852
Среднее значение F1-score на валидационных фолдах: 0.793


Результаты примерно аналогичные, мы наблюдаем небольшое переобучение на тренировочных и валидационных фолдах, которое хотелось бы убрать. При этом модель хорошо справляется с классификацией, так как даже на случайных параметрах модель имеет метрику близкую к лучшей модели логистической регрессии.

In [ ]:
param_grid_forest = {'n_estimators' : list(range(100, 1001, 200)),
                     'criterion' : ['gini', 'entropy'],
                     'max_depth' : list(range(1, 31, 2)),
                     'max_features' : list(range(100, 1701, 200)),
                     'min_samples_leaf' : list(range(1, 31, 2))
                     }

random_search_forest = RandomizedSearchCV(
    estimator=random_forest,
    param_distributions=param_grid_forest,
    cv=kf,
    scoring='f1',
    return_train_score=True,
    n_iter=20
)
%time random_search_forest.fit(X_train, y_train)

In [ ]:
# Получаем полные данные о каждой модели
results_G_S_1 = pd.DataFrame(grid_search_1.cv_results_)
best_index = grid_search_1.best_index_

In [ ]:
# Получаем полные данные о каждой модели
results_G_S_forest = pd.DataFrame(grid_search_forest.cv_results_)
best_index - grid_search_forest.best_index_
# Выводим результаты
# Считаем метрики
f1_score_train_G_S_forest = np.round(results_G_S_forest.loc[best_index, 'mean_train_score'], 2)
f1_score_test_G_S_forest = np.round(results_G_S_forest.loc[best_index, 'mean_test_score'], 2)
print(f'Среднее F1-score на тренировочных фолдах лучшей модели: {f1_score_train_G_S_forest}')
print(f'Среднее F1-score на валидационных фолдах лучшей модели: {f1_score_test_G_S_forest}')
print(f'Наилучшие значения гиперпараметров: {grid_search_forest.best_params_}')
print(f'F1-score на тестовом наборе: {np.round(grid_search_forest.score(X_test, y_test), 2)}')

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_grid_forest = {'n_estimators' : list(range(100, 1001, 200)),
                     'criterion' : ['gini', 'entropy'],
                     'max_depth' : list(range(1, 31, 2)),
                     'max_features' : list(range(100, 1701, 200)),
                     'min_samples_leaf' : list(range(1, 31, 2))
                     }

random_search_forest = RandomizedSearchCV(
    estimator=random_forest,
    param_distributions=param_grid_forest,
    cv=kf,
    n_iter=5,
    n_jobs=-1,
    return_train_score=True,
    random_state=42
)
%time random_search_forest.fit(X_train, y_train)

CPU times: total: 10.8 s
Wall time: 4min 20s


,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'criterion': ['gini', 'entropy'], 'max_depth': [1, 3, ...], 'max_features': [100, 300, ...], 'min_samples_leaf': [1, 3, ...], ...}"
,n_iter,5
,scoring,None
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [ ]:
results_R_S_forest = pd.DataFrame(random_search_forest.cv_results_)
best_index = random_search_forest.best_index_

f1_score_train_R_S_forest = results_R_S_forest.loc[best_index, 'mean_train_score']
f1_score_test_R_S_forest = results_R_S_forest.loc[best_index, 'mean_test_score']
print(f'Среднее F1-score на тренировочных фолдах лучшей модели: {f1_score_train_R_S_forest}')
print(f'Среднее F1-score на тестовых фолдах лучшей модели: {f1_score_test_R_S_forest}')
print(f'Лучшие гиперпараметры модели: {random_search_forest.best_params_}')
print(f'F1-score на тестовом наборе: {np.round(random_search_forest.score(X_test, y_test), 2)}')

Среднее F1-score на тренировочных фолдах лучшей модели: 0.8421904761904762
Среднее F1-score на тестовых фолдах лучшей модели: 0.7756190476190477
Лучшие гиперпараметры модели: {'n_estimators': 100, 'min_samples_leaf': 19, 'max_features': 700, 'max_depth': 17, 'criterion': 'entropy'}
F1-score на тестовом наборе: 0.79


In [ ]:
# Получаем полные данные о каждой модели
results_R_S_1 = pd.DataFrame(random_search_1.cv_results_)
best_index = random_search_1.best_index_

f1_score_train_R_S_1 = np.round(results_R_S_1.loc[best_index, 'mean_train_score'], 2)
f1_score_test_R_S_1 = np.round(results_R_S_1.loc[best_index, 'mean_test_score'], 2)
print(f'Среднее F1-score на тренировочных фолдах лучшей модели: {f1_score_train_G_S_1}')
print(f'Среднее F1-score на тестовых фолдах лучшей модели: {f1_score_test_G_S_1}')
print(f'Лучшие гиперпараметры модели: {random_search_1.best_params_}')
print(f'F1-score на тестовом наборе: {np.round(random_search_1.score(X_test, y_test), 2)}')